In [1]:
#format the book
from __future__ import division, print_function
%matplotlib inline
import sys
sys.path.insert(0, '..')
import book_format
book_format.set_style()

# 將多變量方程式轉換為單變量情形 Converting the Multivariate Equations to the Univariate Case

multivariate Kalman filter（多變量卡爾曼濾波器）的方程式看起來和 univariate（單變量）濾波器的方程式並不相像。然而，如果我們使用一維的 state（狀態）與 measurement（量測），這些方程式確實會化簡為單變量的方程式。這一節會讓你對 Kalman filter 方程式實際在做什麼有深刻的 intuition（直覺）。雖然不讀這一節也能理解本書其餘部分，但我建議仔細閱讀，因為它應該能讓其餘內容更容易理解。

以下是 prediction（預測）的多變量方程式。 

$$
\begin{aligned}
\mathbf{\bar{x}} &= \mathbf{F x} + \mathbf{B u} \\
\mathbf{\bar{P}} &= \mathbf{FPF}^\mathsf{T} + \mathbf Q
\end{aligned}
$$

對於單變量問題，狀態 $\mathbf x$ 只有一個變數，所以它是 $1\times 1$ 的 matrix（矩陣）。我們的運動 $\mathbf{u}$ 也是 $1\times 1$ 矩陣。因此，$\mathbf{F}$ 和 $\mathbf B$ 也必須是 $1\times 1$ 矩陣。這表示它們全都是純量，我們可以寫成

$$\bar{x} = Fx + Bu$$

這裡的變數沒有用粗體，表示它們不是矩陣或 vector（向量）。 

我們的 state transition（狀態轉移）很簡單──下一個狀態與目前狀態相同，所以 $F=1$。運動轉移也一樣，所以 $B=1$。於是我們得到

$$x = x + u$$

這等同於上一章的 Gaussian（高斯）方程式

$$ \mu = \mu_1+\mu_2$$

希望整體流程已經清楚了，所以接下來的部分我會講快一點。我們有

$$\mathbf{\bar{P}} = \mathbf{FPF}^\mathsf{T} + \mathbf Q$$

同樣地，由於我們的狀態只有一個變數，$\mathbf P$ 和 $\mathbf Q$ 也必須是 $1\times 1$ 矩陣，可以把它們當作純量，得到  

$$\bar{P} = FPF^\mathsf{T} + Q$$

我們已經知道 $F=1$。純量的 transpose（轉置）就是它本身，所以 $F^\mathsf{T} = 1$。這得到

$$\bar{P} = P + Q$$

這等同於高斯方程式 

$$\sigma^2 = \sigma_1^2 + \sigma_2^2$$

這證明了在維度為 1 的情況下，多變量預測方程式所做的數學運算與單變量方程式相同。

以下是 update step（更新步驟）的方程式：

$$
\begin{aligned}
\mathbf{K}&= \mathbf{\bar{P}H}^\mathsf{T} (\mathbf{H\bar{P}H}^\mathsf{T} + \mathbf R)^{-1} \\
\textbf{y} &= \mathbf z - \mathbf{H \bar{x}}\\
\mathbf x&=\mathbf{\bar{x}} +\mathbf{K\textbf{y}} \\
\mathbf P&= (\mathbf{I}-\mathbf{KH})\mathbf{\bar{P}}
\end{aligned}
$$

和上面一樣，所有矩陣都變成純量。$H$ 定義了如何從位置轉換為量測值。兩者都是位置，所以不需要轉換，因此 $H=1$。我們一步到位地代入已知的值並轉換為純量。1x1 矩陣的 inverse（反矩陣）就是該值的倒數，所以我們把矩陣求反轉換成除法。

$$
\begin{aligned}
K &=\frac{\bar{P}}{\bar{P} + R} \\
y &= z - \bar{x}\\
x &=\bar{x}+Ky \\
P &= (1-K)\bar{P}
\end{aligned}
$$

在繼續證明之前，我希望你看看這些方程式，體會它們實作的是多麼簡單的概念。residual（殘差）$y$ 不過就是量測值減去預測值。gain（增益）$K$ 依據我們對上一次預測的確信程度與對量測值的確信程度來調整比例。我們以舊的 $x$ 值加上按比例調整後的殘差，來選出新的狀態 $x$。最後，我們根據對量測值的確信程度來更新 uncertainty（不確定性）。就演算法而言，這聽起來應該和我們在上一章做的事完全一樣。

我們來完成代數運算以證明這一點。回想一下，單變量的更新步驟方程式是：

$$
\begin{aligned}
\mu &=\frac{\sigma_1^2 \mu_2 + \sigma_2^2 \mu_1} {\sigma_1^2 + \sigma_2^2}, \\
\sigma^2 &= \frac{1}{\frac{1}{\sigma_1^2} + \frac{1}{\sigma_2^2}}
\end{aligned}
$$

這裡我們令 $\mu_1$ 為狀態 $x$，$\mu_2$ 為量測值 $z$。由此可知 $\sigma_1^2$ 是狀態的不確定性 $P$，而 $\sigma_2^2$ 是 measurement noise（量測雜訊）$R$。我們把它們代入。

$$\begin{aligned} \mu &= \frac{Pz + Rx}{P+R} \\
\sigma^2 &= \frac{1}{\frac{1}{P} + \frac{1}{R}}
\end{aligned}$$

我先處理 $\mu$。多變量情形中對應的方程式是

$$
\begin{aligned}
x &= x + Ky \\
&= x + \frac{P}{P+R}(z-x) \\
&= \frac{P+R}{P+R}x + \frac{Pz - Px}{P+R} \\
&= \frac{Px + Rx + Pz - Px}{P+R} \\
&= \frac{Pz + Rx}{P+R}
\end{aligned}
$$

現在來看 $\sigma^2$。多變量情形中對應的方程式是

$$ 
\begin{aligned}
P &= (1-K)P \\
&= (1-\frac{P}{P+R})P \\
&= (\frac{P+R}{P+R}-\frac{P}{P+R})P \\
&= (\frac{P+R-P}{P+R})P \\
&= \frac{RP}{P+R}\\
&= \frac{1}{\frac{P+R}{RP}}\\
&= \frac{1}{\frac{R}{RP} + \frac{P}{RP}} \\
&= \frac{1}{\frac{1}{P} + \frac{1}{R}}
\quad\blacksquare
\end{aligned}
$$

我們已經證明，當只有一個 state variable（狀態變數）時，多變量方程式等同於單變量方程式。在結束這一節之前，我要承認一個小瑕疵──我對 $H=1$ 與 $F=1$ 的斷言是含糊帶過的。一般而言，我們知道這並不成立。例如，數位溫度計可能以伏特為單位提供量測值，而我們需要把它轉換成溫度，這個轉換就是用 $H$ 來做。我略過了這個問題，好讓說明盡可能簡單、精簡。要把這種一般化加進上面的方程式、重做代數運算並仍得到相同的結果，是非常直接的。\\\